# Task 3 — Gender G2 five-fold confirmation

Run All reuses completed G2 folds 0 and 4. It trains only folds 1, 2, and 3 with the exact frozen translation contract. No other experiment runs.


## 1. Colab and repository setup

Use a GPU runtime. Artifacts and the registry are stored in Drive.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin task-3-gender-usage-classification
$ git switch task-3-gender-usage-classification
$ git merge --ff-only origin/task-3-gender-usage-classification
Repository ready: /content/MLA2
Commit: cb6bc79fbc81ab4c98b7bd2c3e4a56fe3461aefe


## 2. Restore the official teacher data

This uses only the Task 3 teacher ZIP. High-resolution public data is excluded.


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


Teacher data ready: 44,441 images


## 3. Frozen confirmation rule

The two-fold screen suggested that ±2-pixel training translation may improve Gender. Folds 1–3 are fresh confirmation folds. They must remain separate from the screen result, and the final decision must also inspect the pooled five-fold result, every fold, every class, calibration, robustness, time, and memory.


In [3]:
GENDER_E6_PARENT_RUN_IDS = (
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f0_s2753_a8c09286451b_20260831T090059Z0bab1f",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f1_s2753_a8c09286451b_20260831T090940Z6e10b5",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f2_s2753_a8c09286451b_20260831T091823Zabb677",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f3_s2753_a8c09286451b_20260831T092710Zee7c6a",
    "t3_gender_e6_gem_p3_gender_smallcnngem3_f4_s2753_a8c09286451b_20260831T093553Z63b5fd",
)

from fashion.train.task3_dataset_v2 import check_task3_gender_v2_g2_confirmation_setup
from fashion.train.task3_experiments import latest_completed_gender_e6_parent_run_ids

resolved = latest_completed_gender_e6_parent_run_ids(output_root=DRIVE_TASK_DIR)
if resolved != GENDER_E6_PARENT_RUN_IDS:
    raise RuntimeError("The resolved E6 parents differ from the frozen tuple.")
preflight = check_task3_gender_v2_g2_confirmation_setup(
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    output_root=DRIVE_TASK_DIR,
    root=REPO_DIR,
    device_name="cuda",
)
print("GPU:", preflight["environment"]["gpu"])
print("Already complete:", preflight["completed_screen_fold_run_ids"])
print("Folds this notebook may train:", preflight["confirmation_folds_to_train"])
print("Optimizer steps during check:", preflight["optimizer_steps"])


GPU: NVIDIA L4
Already complete: {'0': 't3_gender_v2_g2_translation_gender_smallcnngem3_f0_s2753_cb072542dbdc_20260904T135102Z6698e6', '4': 't3_gender_v2_g2_translation_gender_smallcnngem3_f4_s2753_cb072542dbdc_20260904T140011Z2211b1'}
Folds this notebook may train: [1, 2, 3]
Optimizer steps during check: 0


## 4. Complete G2

This cell trains folds 1–3 only. Matching completed runs are reused after a disconnect. It saves a fresh-fold aggregate and a separate five-fold aggregate without replacing the original screen aggregate.


In [4]:
from fashion.train.task3_dataset_v2 import run_task3_gender_v2_g2_confirmation
from fashion.train.task3_experiments import audit_completed_registry_rows

g2 = run_task3_gender_v2_g2_confirmation(
    parent_run_ids=GENDER_E6_PARENT_RUN_IDS,
    output_root=DRIVE_TASK_DIR,
    folds=(1, 2, 3),
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    device_name="cuda",
    reuse_completed=True,
)
audit = audit_completed_registry_rows(DRIVE_REGISTRY, g2["fold_run_ids"])
{
    "trained_folds": g2["trained_folds"],
    "reused_screen_folds": g2["reused_screen_folds"],
    "fresh_macro_f1": g2["fresh"]["metrics"]["macro_f1"],
    "five_fold_macro_f1": g2["all_five"]["metrics"]["macro_f1"],
    "five_fold_metrics": g2["all_five"]["metrics_path"],
    "registry_audit": audit,
}


[task3] preparing target=gender fold=1: train=26,217 (before selection=26,217), validation=6,556
[task3] fitting fold-training RGB statistics for target=gender fold=1
[task3] RGB statistics ready for target=gender fold=1
[task3] registered t3_gender_v2_g2_translation_gender_smallcnngem3_f1_s2753_cb072542dbdc_20260904T153837Z5db37d; the first optimiser step may now run
[task3] target=gender fold=1 epoch=1/30 train_loss=0.5886 train_macro_f1=0.4611 validation_loss=0.5175 validation_macro_f1=0.5366
[task3] target=gender fold=1 epoch=2/30 train_loss=0.4375 train_macro_f1=0.6198 validation_loss=0.4104 validation_macro_f1=0.6161
[task3] target=gender fold=1 epoch=3/30 train_loss=0.3853 train_macro_f1=0.6640 validation_loss=0.3932 validation_macro_f1=0.5937
[task3] target=gender fold=1 epoch=4/30 train_loss=0.3544 train_macro_f1=0.6988 validation_loss=0.4133 validation_macro_f1=0.6419
[task3] target=gender fold=1 epoch=5/30 train_loss=0.3231 train_macro_f1=0.7249 validation_loss=0.4533 valida

{'trained_folds': [1, 2, 3],
 'reused_screen_folds': [0, 4],
 'fresh_macro_f1': 0.7532042118897733,
 'five_fold_macro_f1': 0.7502173957741934,
 'five_fold_metrics': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_v2_g2_translation/gender/aggregate_five_fold/metrics.json',
 'registry_audit': {'registry_path': '/content/drive/MyDrive/MLA2/task3/results/runs.csv',
  'run_ids': ['t3_gender_v2_g2_translation_gender_smallcnngem3_f0_s2753_cb072542dbdc_20260904T135102Z6698e6',
   't3_gender_v2_g2_translation_gender_smallcnngem3_f1_s2753_cb072542dbdc_20260904T153837Z5db37d',
   't3_gender_v2_g2_translation_gender_smallcnngem3_f2_s2753_cb072542dbdc_20260904T154745Z7d81aa',
   't3_gender_v2_g2_translation_gender_smallcnngem3_f3_s2753_cb072542dbdc_20260904T155700Zb0abef',
   't3_gender_v2_g2_translation_gender_smallcnngem3_f4_s2753_cb072542dbdc_20260904T140011Z2211b1'],
  'completed_rows': 5,
  'ready': True}}

## 5. Stop

Do not start another experiment here. Analyse the saved G2 confirmation first.
